# Give a model a fair exam

Imagine sorting messages into routine and urgent. A guess that says "routine" every time might get many answers right but miss every urgent message. In this notebook, **baseline** means that simple guess; the trained rule gets a chance to beat it. We use bundled teaching data, not real messages or patients.

Before running: should we let the model practice on the held-out exam examples? **No.** The code below separates training examples from test examples first and fits its scale-adjustment step only on training data. Read [the ML lesson](../../docs/03-machine-learning.md) for the small inbox example. No downloads or keys are needed after installation.

In [ ]:
import sklearn
from sklearn.datasets import load_breast_cancer
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score, confusion_matrix
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
print('scikit-learn:', sklearn.__version__)
dataset = load_breast_cancer()
train_x, test_x, train_y, test_y = train_test_split(dataset.data, dataset.target, test_size=0.2, random_state=42, stratify=dataset.target)
baseline = DummyClassifier(strategy='most_frequent').fit(train_x, train_y)
model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(model, train_x, train_y, scoring='balanced_accuracy', cv=folds)
model.fit(train_x, train_y)
baseline_score = balanced_accuracy_score(test_y, baseline.predict(test_x))
test_score = balanced_accuracy_score(test_y, model.predict(test_x))
assert train_x.shape[0] + test_x.shape[0] == len(dataset.target)
assert scores.shape == (5,) and 0 <= test_score <= 1
print('baseline:', baseline_score, 'CV mean:', scores.mean(), 'held-out:', test_score)
print('confusion matrix:', confusion_matrix(test_y, model.predict(test_x)))

## Check
Why is `StandardScaler` inside the pipeline? It must learn each fold's mean and scale from the *training fold only*. This example's split is not grouped by patient or hospital, so it cannot establish clinical validity. Try a decision tree and compare CV rather than selecting by the held-out test result.